In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


In [2]:
df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test.csv'))
display(df_test.head())

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW
0,siteD,2020-01-01 00:00:00,24.70,0,4.8
1,siteD,2020-01-01 00:15:00,24.61,0,4.8
2,siteD,2020-01-01 00:30:00,24.53,0,4.8
3,siteD,2020-01-01 00:45:00,24.45,0,4.8
4,siteD,2020-01-01 01:00:00,24.36,0,4.8


In [ ]:
def preprocess_data(df_in):
    df = df_in.copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2]).astype(int)
    df['Is_Spring'] = df['Month'].isin([3, 4, 5]).astype(int)
    df['Is_Fall'] = df['Month'].isin([9, 10, 11]).astype(int)
    # Create hour of day categories
    df['Morning'] = ((df['Hour'] >= 5) & (df['Hour'] < 11)).astype(int)
    df['Afternoon'] = ((df['Hour'] >= 11) & (df['Hour'] < 18)).astype(int)
    df['Evening'] = ((df['Hour'] >= 18) & (df['Hour'] < 24)).astype(int)
    df['Nighttime'] = ((df['Hour'] >= 0) & (df['Hour'] < 5)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site'], inplace=True)
    return df

df = preprocess_data(df_test)

In [4]:
# Define neural network
class Net(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, num_classes)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Load the trained model from file
input_dim = df.shape[1]# Number of features
num_classes = 3  # Number of classes in target variable
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('trained_model.pth'))

# load the scaler
scaler = joblib.load('scaler.pkl')

In [5]:
# Convert DataFrame to torch tensor
X = scaler.transform(df.values)
X = torch.tensor(X, dtype=torch.float32)

# Set model to evaluation mode
model_loaded.eval()
with torch.no_grad():
    outputs = model_loaded(X)
    predictions = torch.argmax(outputs, dim=1)

# convert 2 in predictions to -1
predictions = np.where(predictions == 2, -1, predictions)

# Calculate the distribution of predictions classes
unique, counts = np.unique(predictions, return_counts=True)
distribution_pred = dict(zip(unique, counts))
print(distribution_pred)

{np.int64(-1): np.int64(2206), np.int64(0): np.int64(32023), np.int64(1): np.int64(811)}


In [6]:
df_submission = df_test[['Site', 'Timestamp_Local']].copy(deep=True)
df_submission['Demand_Response_Flag'] = predictions
df_submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")

Submission file created: submission.csv


In [7]:
df_submission.head()

,Site,Timestamp_Local,Demand_Response_Flag
0,siteD,2020-01-01 00:00:00,0
1,siteD,2020-01-01 00:15:00,0
2,siteD,2020-01-01 00:30:00,0
3,siteD,2020-01-01 00:45:00,0
4,siteD,2020-01-01 01:00:00,0


In [8]:
df_submission.shape

(35040, 3)